# Module 11: Real-Time Analytics with Spark Streaming

**Aligned with Techcombank JD**: *"Real-time analytics platform for critical decision making"*

## Topics
1. Structured Streaming fundamentals
2. Real-time transaction monitoring
3. Streaming aggregations
4. Exactly-once processing
5. Production patterns

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pathlib import Path
import shutil

spark = SparkSession.builder.appName("Module11-Streaming").master("local[*]").config("spark.sql.shuffle.partitions", "4").getOrCreate()
DATA_RAW = Path("../data/raw")
STREAM_INPUT = Path("../data/streaming/input")
STREAM_OUTPUT = Path("../data/streaming/output")
CHECKPOINT = Path("../data/streaming/checkpoint")

---
## 1. Structured Streaming Basics

**Key Concept**: Treat streaming data as an unbounded table

```
Input Stream → Spark Streaming → Output Sink
(Kafka, Files)   (Transformations)  (Delta, Kafka, Console)
```

In [ ]:
# Define schema (required for streaming)
txn_schema = StructType([
    StructField("txn_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("txn_datetime", StringType(), True),
    StructField("txn_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("status", StringType(), True),
    StructField("reference", StringType(), True),
    StructField("description", StringType(), True)
])

print("Schema defined for streaming")

In [ ]:
# Prepare streaming input directory (simulate streaming with file source)
STREAM_INPUT.mkdir(parents=True, exist_ok=True)
STREAM_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT.mkdir(parents=True, exist_ok=True)

# Copy sample data to simulate incoming files
sample_txn = spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, schema=txn_schema).limit(1000)
sample_txn.write.mode("overwrite").csv(str(STREAM_INPUT / "batch1"), header=True)
print(f"✅ Created sample streaming input: {STREAM_INPUT}")

---
## 2. Read Stream

In [ ]:
# Create streaming DataFrame
stream_df = spark.readStream.format("csv").option("header", "true").schema(txn_schema).load(str(STREAM_INPUT))

print(f"Is streaming: {stream_df.isStreaming}")

---
## 3. Real-Time Transaction Monitoring

**Use Case**: Detect suspicious transactions in real-time

In [ ]:
# Transform: Flag high-value transactions
monitored_stream = stream_df.withColumn(
    "txn_timestamp", to_timestamp(col("txn_datetime"))
).withColumn(
    "is_high_value", col("amount") > 100_000_000
).withColumn(
    "is_suspicious", 
    (col("amount") > 500_000_000) | (col("status") == "Failed")
)

In [ ]:
# Write to console (for testing)
query = monitored_stream.filter(col("is_suspicious")).select(
    "txn_id", "account_id", "amount", "status", "is_suspicious"
).writeStream.format("console").outputMode("append").option("truncate", "false").start()

# Let it run briefly
import time
time.sleep(5)
query.stop()
print("✅ Stream query stopped")

---
## 4. Streaming Aggregations (Windowed)

**Use Case**: Real-time dashboard showing transactions per minute by channel

In [ ]:
# Windowed aggregation
stream_with_ts = stream_df.withColumn("txn_timestamp", to_timestamp(col("txn_datetime")))

# Group by 1-minute windows
windowed_agg = stream_with_ts.withWatermark("txn_timestamp", "10 minutes").groupBy(
    window(col("txn_timestamp"), "1 minute"),
    col("channel")
).agg(
    count("txn_id").alias("txn_count"),
    sum("amount").alias("total_amount")
)

print("Windowed aggregation defined (1-minute tumbling windows)")

In [ ]:
# Write aggregated results
agg_query = windowed_agg.writeStream.format("console").outputMode("update").option("truncate", "false").start()

time.sleep(5)
agg_query.stop()

---
## 5. Production Patterns

### Checkpointing (Exactly-Once)

In [ ]:
# Write to Parquet with checkpointing
output_query = monitored_stream.writeStream \
    .format("parquet") \
    .option("path", str(STREAM_OUTPUT / "transactions")) \
    .option("checkpointLocation", str(CHECKPOINT / "txn_checkpoint")) \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .start()

time.sleep(15)
output_query.stop()
print(f"✅ Output written to: {STREAM_OUTPUT}")

### Kafka Integration Pattern (Production)

In [ ]:
# Kafka read pattern (not executed - requires Kafka)
kafka_read_pattern = """
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "latest") \
    .load()

# Parse JSON from Kafka value
parsed = kafka_df.select(
    from_json(col("value").cast("string"), txn_schema).alias("data")
).select("data.*")
"""
print(kafka_read_pattern)

---
## Streaming Best Practices

✅ **Always define schema** - no inferSchema for streaming
✅ **Use checkpointing** - ensures exactly-once processing
✅ **Set watermarks** - for handling late data
✅ **Monitor with metrics** - track throughput, latency
✅ **Test with rate source** - simulate streaming in dev
✅ **Separate streaming clusters** - isolate from batch workloads

In [ ]:
# Cleanup
shutil.rmtree(STREAM_INPUT, ignore_errors=True)
shutil.rmtree(STREAM_OUTPUT, ignore_errors=True)
shutil.rmtree(CHECKPOINT, ignore_errors=True)
spark.stop()